# Deliverable 2 - Fold-Safe 5-Fold Implementation

This notebook documents and reruns the corrected Deliverable 2 implementation. The key correction is that feature files are now materialized per fold and per split:

- `results/fold_features/Fold_i/training_feature_matrix_curated.csv`
- `results/fold_features/Fold_i/validation_feature_matrix_curated.csv`
- `results/fold_features/Fold_i/test_feature_matrix_curated.csv`

Model preprocessing and feature selection are fit inside each fold. The test split is used only for final held-out evaluation.

## What Changed

1. The old global `subject_feature_matrix.csv` is no longer used by the Deliverable 2 runner.
2. Each fold extracts training, validation, and test feature matrices from that fold's own signal files.
3. Random Forest, Linear-SVM, RBF-SVM, 1D-ResNet, LSTM, GRU, and PatchTST all consume feature vectors.
4. Reported metrics include Accuracy, Sensitivity, Specificity, and AUC for every model/scope.
5. The enhanced feature set includes TSFresh, left-right asymmetry, gait timing/COP, and wavelet features.

In [ ]:
from pathlib import Path
import subprocess
import sys

import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd()
TABLES = ROOT / 'results' / 'tables'
FIGURES = ROOT / 'results' / 'figures'
FOLD_FEATURES = ROOT / 'results' / 'fold_features'

assert (ROOT / 'scripts' / 'run_deliverable2_foldsafe.py').exists(), 'Run this notebook from the Data project directory.'

## Optional: Rerun the Full Pipeline

The pipeline has already been run once. Set `RUN_PIPELINE = True` to regenerate all fold-local feature matrices and model results. Keep `--force-features` off unless you intentionally want to recompute TSFresh features from every split signal.

In [ ]:
RUN_PIPELINE = False

if RUN_PIPELINE:
    cmd = [
        sys.executable,
        'scripts/run_deliverable2.py',
        '--feature-set', 'curated',
        '--selection-metric', 'f1',
        '--max-epochs', '6',
        '--patience', '2',
        '--batch-size', '16',
        '--deep-k-features', '200',
    ]
    subprocess.run(cmd, check=True)

## Fold-Safe Feature Extraction Audit

There should be 15 feature files: 5 folds x 3 splits. Each row below is created from one split directory, not from a single all-subject feature matrix.

In [ ]:
audit = pd.read_csv(TABLES / 'deliverable2_fold_safe_feature_audit.csv')
display(audit)

assert audit.shape[0] == 15
assert set(audit['split']) == {'training', 'validation', 'test'}
assert audit.groupby('fold')['split'].nunique().eq(3).all()

## Model Summary

The summary is averaged across the five held-out test folds. The main required metrics are Accuracy, Sensitivity, Specificity, and AUC.

In [ ]:
summary = pd.read_csv(TABLES / 'deliverable2_model_summary.csv')
metric_cols = [
    'family', 'scope', 'model',
    'test_accuracy_mean', 'test_sensitivity_mean', 'test_specificity_mean', 'test_auc_mean',
    'test_precision_mean', 'test_f1_mean',
]
display(summary.sort_values('test_accuracy_mean', ascending=False)[metric_cols].head(15))

## Best Model Per Feature View

In [ ]:
best = pd.read_csv(TABLES / 'deliverable2_best_models_by_scope.csv')
display(best[['scope', 'family', 'model', 'test_accuracy_mean', 'test_sensitivity_mean', 'test_specificity_mean', 'test_auc_mean']])

## Figures

In [ ]:
display(Image(filename=str(FIGURES / 'deliverable2_metric_summary.png')))
display(Image(filename=str(FIGURES / 'deliverable2_best_confusion_matrices.png')))

## Research-Informed Feature Choices

The implemented enhanced feature vector follows the feature directions repeatedly used with this PhysioNet gait dataset:

- PhysioNet describes the dataset as 16 vertical ground reaction force sensors plus total-force channels, and explicitly notes that center-of-pressure, stride timing, swing timing, and stride-to-stride variability can be derived from the signals.
- Alam et al. (PLOS ONE, 2017) report that useful features include swing/stride variability, peak force, and center-of-pressure measures, with SVM-style feature selection and classifiers.
- Public GitHub projects on this dataset commonly compare Random Forest, linear/RBF SVM, and related classical models after statistical compression of the time series.
- CNN/time-frequency projects motivate the wavelet branch; here the CWT idea is represented as fold-safe Morlet wavelet summary features rather than image-based CNN input.

Implemented from these ideas: gait timing statistics, coefficient-of-variation style statistics, COP summaries, left-right asymmetry, and wavelet power summaries.

In [ ]:
research_notes = pd.read_csv(TABLES / 'deliverable2_research_feature_notes.csv')
display(research_notes)

## Current Main Result

The strongest fold-safe result is usually the enhanced Linear-SVM row. It is close to the professor's reported Linear-SVM result while preserving the stricter split-local feature extraction protocol.

In [ ]:
enhanced_linear = summary[(summary['scope'] == 'enhanced') & (summary['model'] == 'linear_svm')].iloc[0]
enhanced_linear[['test_accuracy_mean', 'test_sensitivity_mean', 'test_specificity_mean', 'test_auc_mean', 'test_precision_mean', 'test_f1_mean']]